In [1]:
import math
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm.auto import tqdm
import wandb
import timm

sys.path.append(".")
from src.evit.evit import EViT
from src.evit.helpers import adjust_keep_rate, complement_idx
from src.evit.utils import MetricLogger, SmoothedValue
from src.dataset import HistologicalImageDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


/home/oem/EntropyPruning/src/evit/evit.py:638: UserWarning: Overwriting deit_tiny_patch16_224 in registry with src.evit.evit.deit_tiny_patch16_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  def deit_tiny_patch16_224(pretrained=False, **kwargs):
/home/oem/EntropyPruning/src/evit/evit.py:648: UserWarning: Overwriting deit_small_patch16_224 in registry with src.evit.evit.deit_small_patch16_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  def deit_small_patch16_224(pretrained=False, **kwargs):
/home/oem/EntropyPruning/src/evit/evit.py:728: UserWarning: Overwriting deit_base_patch16_224 in registry with src.evit.evit.deit_base_patch16_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  def deit_base_patch16_224(pretrained=False, **kwargs):
/home/oem/EntropyPruning/src/evit/evit.py:7

## 1 · Configurazione

In [2]:
BACKBONE = "hf-hub:MahmoodLab/uni"

CFG = dict(
    # ---- Dati ----
    data_dir        = "/data/NCT-CRC-HE",
    img_size        = 224,
    batch_size      = 8,
    num_workers     = 4,

    # ---- Backbone ----
    backbone        = BACKBONE,
    patch_size      = 16,
    embed_dim       = 1024,
    depth           = 24,
    num_heads       = 16,
    mlp_ratio       = 4.0,
    drop_path       = 0.2,
    global_pool     = "avg",

    # ---- EViT ----
    base_keep_rate  = 0.7,
    drop_loc        = (8, 16, 22),
    fuse_token      = True,
    keep_rate_warmup_epochs = 5,

    # ---- Ottimizzazione ----
    lr              = 4e-3,
    min_lr          = 1e-6,
    layer_decay     = 0.75,
    weight_decay    = 0.05,
    epochs          = 50,
    warmup_epochs   = 5,
    accum_steps     = 2,
    max_norm        = 1.0,
    label_smoothing = 0.1,

    seed            = 42,
)

CFG["dataset_name"] = Path(CFG["data_dir"]).name
CFG["output_dir"]   = Path(
    f"/data/checkpoints-Attention-Pruning/{CFG['dataset_name']}"
    f"/uni_evit_{BACKBONE}_kr{CFG['base_keep_rate']}"
)
CFG["output_dir"].mkdir(parents=True, exist_ok=True)

_keep_rate = [1.0] * CFG["depth"]
for loc in CFG["drop_loc"]:
    _keep_rate[loc] = CFG["base_keep_rate"]
CFG["keep_rate"] = tuple(_keep_rate)

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
print(f"Output dir  : {CFG['output_dir']}")
print(f"Keep rate   : {CFG['keep_rate']}")

Output dir  : /data/checkpoints-Attention-Pruning/NCT-CRC-HE/uni_evit_hf-hub:MahmoodLab/uni_kr0.7
Keep rate   : (1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7, 1.0)


## 2 · Dati & Augmentation

In [3]:
train_tf = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation((90, 90))], p=0.5),
    T.RandomApply([T.ColorJitter(0.2, 0.2, 0.1, 0.05)], p=0.5),
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
eval_tf = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

train_ds = HistologicalImageDataset(f"{CFG['data_dir']}/train", transform=train_tf)
val_ds   = HistologicalImageDataset(f"{CFG['data_dir']}/val",   transform=eval_tf)
test_ds  = HistologicalImageDataset(f"{CFG['data_dir']}/test",  transform=eval_tf)

counts  = np.bincount(train_ds.labels)
weights = torch.from_numpy((1.0 / counts)[train_ds.labels]).double()
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"],
          pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_ds, sampler=sampler, drop_last=True, **kw)
val_loader   = DataLoader(val_ds,  shuffle=False, **kw)
test_loader  = DataLoader(test_ds, shuffle=False, **kw)

CLASS_NAMES = train_ds.class_names
N_CLASSES   = len(CLASS_NAMES)
print(f"Classi ({N_CLASSES}): {CLASS_NAMES}")
print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}")

Loading from /data/NCT-CRC-HE/train...


Loading dataset from disk:   0%|          | 0/26 [00:00<?, ?it/s]

Loaded 90000 samples, 9 classes
Class distribution:
  ADI: 9366 (10.4%)
  BACK: 9510 (10.6%)
  DEB: 10361 (11.5%)
  LYM: 10401 (11.6%)
  MUC: 8006 (8.9%)
  MUS: 12182 (13.5%)
  NORM: 7887 (8.8%)
  STR: 9402 (10.4%)
  TUM: 12885 (14.3%)
Loading from /data/NCT-CRC-HE/val...
Loaded 10000 samples, 9 classes
Class distribution:
  ADI: 1041 (10.4%)
  BACK: 1056 (10.6%)
  DEB: 1151 (11.5%)
  LYM: 1156 (11.6%)
  MUC: 890 (8.9%)
  MUS: 1354 (13.5%)
  NORM: 876 (8.8%)
  STR: 1044 (10.4%)
  TUM: 1432 (14.3%)
Loading from /data/NCT-CRC-HE/test...
Loaded 7180 samples, 9 classes
Class distribution:
  ADI: 1338 (18.6%)
  BACK: 847 (11.8%)
  DEB: 339 (4.7%)
  LYM: 634 (8.8%)
  MUC: 1035 (14.4%)
  MUS: 592 (8.2%)
  NORM: 741 (10.3%)
  STR: 421 (5.9%)
  TUM: 1233 (17.2%)
Classi (9): ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
Train: 90000  |  Val: 10000  |  Test: 7180


## 3 · Costruzione Modello

In [4]:
# ── Funzioni per iniezione LayerScale (necessarie per UNI) ──────────────────
from timm.models.vision_transformer import LayerScale
from types import MethodType


def _block_forward_with_ls(self, x, keep_rate=None, tokens=None, get_idx=False):
    """Forward patch per EViT.Block che include ls1/ls2 (layer-scale di UNI)."""
    if keep_rate is None:
        keep_rate = self.keep_rate
    B, N, C = x.shape
    tmp, index, idx, cls_attn, left_tokens = self.attn(self.norm1(x), keep_rate, tokens)
    x = x + self.drop_path(self.ls1(tmp))
    if index is not None:
        non_cls  = x[:, 1:]
        x_others = torch.gather(non_cls, dim=1, index=index)
        if self.fuse_token:
            compl         = complement_idx(idx, N - 1)
            non_topk      = torch.gather(non_cls, dim=1,
                                         index=compl.unsqueeze(-1).expand(-1, -1, C))
            non_topk_attn = torch.gather(cls_attn, dim=1, index=compl)
            extra_token   = (non_topk * non_topk_attn.unsqueeze(-1)).sum(dim=1, keepdim=True)
            x = torch.cat([x[:, :1], x_others, extra_token], dim=1)
        else:
            x = torch.cat([x[:, :1], x_others], dim=1)
    x = x + self.drop_path(self.ls2(self.mlp(self.norm2(x))))
    n_tokens = x.shape[1] - 1
    if get_idx and index is not None:
        return x, n_tokens, idx
    return x, n_tokens, None


def inject_layer_scale(evit_model, uni_state_dict, init_values=1e-5):
    """Inietta LayerScale nei blocchi EViT e carica i gamma da UNI."""
    dim = evit_model.blocks[0].attn.proj.out_features
    for i, blk in enumerate(evit_model.blocks):
        blk.ls1 = LayerScale(dim, init_values=init_values)
        blk.ls2 = LayerScale(dim, init_values=init_values)
        blk.ls1.gamma.data.copy_(uni_state_dict[f"blocks.{i}.ls1.gamma"])
        blk.ls2.gamma.data.copy_(uni_state_dict[f"blocks.{i}.ls2.gamma"])
        blk.forward = MethodType(_block_forward_with_ls, blk)
    print(f"LayerScale iniettati in {len(evit_model.blocks)} blocchi.")


def load_pretrained_into_evit(evit_model: EViT, backbone: str) -> None:
    """
    Carica i pesi pre-addestrati in evit_model usando strict=False.
    I layer EViT-specifici non hanno parametri propri — la struttura
    parametrica è identica al ViT standard.
    """
    print(f"Caricamento pesi backbone '{backbone}'...")
    if backbone == "mae":
        src = timm.create_model(
            "vit_large_patch16_224.mae", pretrained=True,
            img_size=CFG["img_size"], patch_size=CFG["patch_size"],
            embed_dim=CFG["embed_dim"], depth=CFG["depth"],
            num_heads=CFG["num_heads"], mlp_ratio=CFG["mlp_ratio"],
            global_pool="avg", num_classes=0, drop_path_rate=CFG["drop_path"],
        )
    elif backbone == "uni" or "MahmoodLab/uni" in backbone:
        src = timm.create_model(
            "hf-hub:MahmoodLab/uni", pretrained=True,
            init_values=1e-5, dynamic_img_size=True, num_classes=0,
        )
    else:
        raise ValueError(f"Backbone non riconosciuto: {backbone}")

    msg = evit_model.load_state_dict(src.state_dict(), strict=False)
    print(f"  Missing  : {len(msg.missing_keys)}")
    print(f"  Unexpected: {len(msg.unexpected_keys)}")
    del src
    torch.cuda.empty_cache()

In [5]:
# ── Costruzione ──────────────────────────────────────────────────────────────
evit = EViT(
    img_size=CFG["img_size"], patch_size=CFG["patch_size"],
    embed_dim=CFG["embed_dim"], depth=CFG["depth"],
    num_heads=CFG["num_heads"], mlp_ratio=CFG["mlp_ratio"],
    qkv_bias=True, drop_path_rate=CFG["drop_path"],
    num_classes=N_CLASSES, keep_rate=CFG["keep_rate"],
    fuse_token=CFG["fuse_token"],
)

# 1) Carica pesi base (Missing=48 ls1/ls2, Unexpected=0)
load_pretrained_into_evit(evit, CFG["backbone"])

# 2) Inietta LayerScale da UNI (porta Missing a 0)
if "uni" in CFG["backbone"] or "MahmoodLab" in CFG["backbone"]:
    src_sd = timm.create_model(
        "hf-hub:MahmoodLab/uni", pretrained=True,
        init_values=1e-5, dynamic_img_size=True, num_classes=0,
    ).state_dict()
    inject_layer_scale(evit, src_sd, init_values=1e-5)
    del src_sd; torch.cuda.empty_cache()

# 3) Re-inizializza head per N_CLASSES
evit.head = nn.Linear(CFG["embed_dim"], N_CLASSES)
nn.init.trunc_normal_(evit.head.weight, std=0.02)
nn.init.zeros_(evit.head.bias)

model = evit.to(device, dtype=torch.bfloat16)

total   = sum(p.numel() for p in model.parameters()) / 1e6
trainbl = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Parametri totali    : {total:.1f}M")
print(f"Parametri trainable : {trainbl:.1f}M")

Caricamento pesi backbone 'hf-hub:MahmoodLab/uni'...
  Missing  : 2
  Unexpected: 48
LayerScale iniettati in 24 blocchi.
Parametri totali    : 303.4M
Parametri trainable : 303.4M


## 4 · WandB

In [6]:
wandb.init(
    project = "uni-evit",
    name    = (f"{CFG['dataset_name']}_{CFG['backbone']}"
               f"_kr{CFG['base_keep_rate']}_fuse{CFG['fuse_token']}"),
    config  = CFG,
    tags    = [CFG["dataset_name"], CFG["backbone"],
               f"keep{CFG['base_keep_rate']}", "evit"],
)

wandb: Currently logged in as: vincenzo-civale (vincenzo-civale-universi-degli-studi-di-firenze) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 5 · Ottimizzatore con LLRD

In [7]:
def param_groups_lrd(model, weight_decay=0.05,
                     no_weight_decay_list=frozenset(), layer_decay=0.75):
    param_group_names, param_groups = {}, {}
    num_layers = len(model.blocks) + 1

    def get_layer_id(name):
        if name in ("cls_token", "pos_embed"):
            return 0
        if name.startswith("patch_embed"):
            return 0
        if name.startswith("blocks."):
            return int(name.split(".")[1]) + 1
        return num_layers

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        wd      = 0.0 if (param.ndim == 1 or name.endswith(".bias")
                          or name in no_weight_decay_list) else weight_decay
        g_decay = "no_decay" if wd == 0.0 else "decay"
        layer_id  = get_layer_id(name)
        lr_scale  = layer_decay ** (num_layers - layer_id)
        group_key = f"layer_{layer_id:02d}_{g_decay}"

        if group_key not in param_groups:
            param_group_names[group_key] = {"lr_scale": lr_scale,
                                            "weight_decay": wd, "params": []}
            param_groups[group_key]      = {"lr_scale": lr_scale,
                                            "weight_decay": wd, "params": []}
        param_group_names[group_key]["params"].append(name)
        param_groups[group_key]["params"].append(param)

    print(f"{'Gruppo':<30} {'lr_scale':>10}  {'#params':>8}")
    print("-" * 54)
    for k, v in sorted(param_group_names.items()):
        print(f"  {k:<28} {v['lr_scale']:>10.4f}  {len(v['params']):>8}")
    return list(param_groups.values())


effective_lr = CFG["lr"] * CFG["batch_size"] * CFG["accum_steps"] / 256.0
print(f"LR effettivo: {effective_lr:.2e}")

param_groups_list = param_groups_lrd(
    model,
    weight_decay         = CFG["weight_decay"],
    no_weight_decay_list = model.no_weight_decay(),
    layer_decay          = CFG["layer_decay"],
)
optimizer = torch.optim.AdamW(param_groups_list, lr=effective_lr,
                               weight_decay=CFG["weight_decay"])

LR effettivo: 2.50e-04
Gruppo                           lr_scale   #params
------------------------------------------------------
  layer_00_decay                   0.0008         1
  layer_00_no_decay                0.0008         3
  layer_01_decay                   0.0010         4
  layer_01_no_decay                0.0010        10
  layer_02_decay                   0.0013         4
  layer_02_no_decay                0.0013        10
  layer_03_decay                   0.0018         4
  layer_03_no_decay                0.0018        10
  layer_04_decay                   0.0024         4
  layer_04_no_decay                0.0024        10
  layer_05_decay                   0.0032         4
  layer_05_no_decay                0.0032        10
  layer_06_decay                   0.0042         4
  layer_06_no_decay                0.0042        10
  layer_07_decay                   0.0056         4
  layer_07_no_decay                0.0056        10
  layer_08_decay                   0.0

## 6 · Loss & Scheduler

In [8]:
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

steps_per_epoch = len(train_loader) // CFG["accum_steps"]
total_steps     = CFG["epochs"] * steps_per_epoch
warmup_steps    = CFG["warmup_epochs"] * steps_per_epoch


def cosine_schedule_with_warmup(base_lr, min_lr, total_steps, warmup_steps):
    schedule = []
    for t in range(total_steps):
        if t < warmup_steps:
            lr = base_lr * t / max(1, warmup_steps)
        else:
            progress = (t - warmup_steps) / max(1, total_steps - warmup_steps)
            lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))
        schedule.append(lr)
    return np.array(schedule)


lr_schedule = cosine_schedule_with_warmup(
    effective_lr, CFG["min_lr"], total_steps, warmup_steps
)
print(f"Steps totali: {total_steps}  |  Warmup steps: {warmup_steps}")

Steps totali: 281250  |  Warmup steps: 28125


## 7 · Training Loop

In [9]:
def train_one_epoch(model, loader, criterion, optimizer,
                    epoch, grad_offset, iters_per_epoch):
    model.train()
    metric_logger = MetricLogger(delimiter="  ")
    metric_logger.add_meter("lr",   SmoothedValue(window_size=1, fmt="{value:.2e}"))
    metric_logger.add_meter("loss", SmoothedValue(window_size=20))
    metric_logger.add_meter("acc",  SmoothedValue(window_size=20))

    header = f"Epoch [{epoch:02d}/{CFG['epochs']}]"
    grad_i = grad_offset
    optimizer.zero_grad()

    for batch_i, (imgs, labels) in enumerate(
            metric_logger.log_every(loader, print_freq=50, header=header)):

        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # LR update (ogni accum_steps)
        if batch_i % CFG["accum_steps"] == 0:
            current_lr = lr_schedule[min(grad_i, len(lr_schedule) - 1)]
            for pg in optimizer.param_groups:
                pg["lr"] = current_lr * pg.get("lr_scale", 1.0)

        # Keep rate dinamico con warmup
        global_iter = (epoch - 1) * iters_per_epoch + batch_i
        current_keep_rate = adjust_keep_rate(
            iters           = global_iter,
            epoch           = epoch - 1,
            warmup_epochs   = CFG["keep_rate_warmup_epochs"],
            total_epochs    = CFG["epochs"],
            ITERS_PER_EPOCH = iters_per_epoch,
            base_keep_rate  = CFG["base_keep_rate"],
            max_keep_rate   = 1.0,
        )
        dyn_keep_rate = tuple(
            current_keep_rate if CFG["keep_rate"][i] < 1.0 else 1.0
            for i in range(CFG["depth"])
        )

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(imgs, keep_rate=dyn_keep_rate)
            loss   = criterion(logits, labels) / CFG["accum_steps"]

        loss.backward()

        if (batch_i + 1) % CFG["accum_steps"] == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_norm"])
            optimizer.step()
            optimizer.zero_grad()
            grad_i += 1

        acc = (logits.detach().argmax(1) == labels).float().mean().item()
        metric_logger.update(
            loss = loss.item() * CFG["accum_steps"],
            acc  = acc,
            lr   = optimizer.param_groups[-1]["lr"],
        )

    # Flush ultimo accum incompleto
    if len(loader) % CFG["accum_steps"] != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_norm"])
        optimizer.step()
        optimizer.zero_grad()

    return (metric_logger.loss.global_avg,
            metric_logger.acc.global_avg,
            grad_i)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    metric_logger = MetricLogger(delimiter="  ")
    for imgs, labels in metric_logger.log_every(loader, print_freq=50, header="Val"):
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(imgs.to(device, non_blocking=True))
        probs = logits.float().softmax(-1)
        acc   = (probs.argmax(1) == labels.to(device)).float().mean().item()
        metric_logger.update(acc=acc, loss=criterion(logits, labels.to(device)).item())
    return metric_logger.loss.global_avg, metric_logger.acc.global_avg

## 8 · Training

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc, best_epoch = 0.0, 0
grad_step       = 0
iters_per_epoch = len(train_loader)

epoch_bar = tqdm(range(1, CFG["epochs"] + 1), desc="Training", unit="epoch")

for epoch in epoch_bar:
    tr_loss, tr_acc, grad_step = train_one_epoch(
        model, train_loader, criterion, optimizer,
        epoch, grad_step, iters_per_epoch,
    )
    vl_loss, vl_acc = evaluate(model, val_loader)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    # Aggiorna la barra con le metriche correnti
    epoch_bar.set_postfix({
        "tr_loss": f"{tr_loss:.4f}",
        "tr_acc":  f"{tr_acc:.4f}",
        "vl_loss": f"{vl_loss:.4f}",
        "vl_acc":  f"{vl_acc:.4f}",
        "best":    f"{best_val_acc:.4f}",
    })

    wandb.log({
        "epoch"      : epoch,
        "train/loss" : tr_loss,
        "train/acc"  : tr_acc,
        "val/loss"   : vl_loss,
        "val/acc"    : vl_acc,
        "lr"         : optimizer.param_groups[-1]["lr"],
    })

    if vl_acc > best_val_acc:
        best_val_acc, best_epoch = vl_acc, epoch
        torch.save(model.state_dict(), CFG["output_dir"] / "best_model.pt")
        wandb.summary["best_val_acc"] = best_val_acc
        wandb.summary["best_epoch"]   = best_epoch
        epoch_bar.write(f"  ✅ Epoch {epoch:02d} — nuovo best val_acc={best_val_acc:.4f}")

epoch_bar.write(f"\nBest val_acc={best_val_acc:.4f} @ epoch {best_epoch}")

Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

Epoch [01/50]  [    0/11250]  eta: 4:26:24  lr: 0.00e+00  loss: 2.6042 (2.6042)  acc: 0.0000 (0.0000)  time: 1.4208  data: 0.3277  max mem: 2824
Epoch [01/50]  [   50/11250]  eta: 0:23:02  lr: 2.22e-07  loss: 2.3744 (2.4893)  acc: 0.1250 (0.0760)  time: 0.0943  data: 0.0001  max mem: 4589
Epoch [01/50]  [  100/11250]  eta: 0:20:13  lr: 4.44e-07  loss: 2.6716 (2.4792)  acc: 0.0000 (0.0817)  time: 0.0940  data: 0.0001  max mem: 4589
Epoch [01/50]  [  150/11250]  eta: 0:19:13  lr: 6.67e-07  loss: 2.5080 (2.4894)  acc: 0.0000 (0.0844)  time: 0.0940  data: 0.0001  max mem: 4589
Epoch [01/50]  [  200/11250]  eta: 0:18:42  lr: 8.89e-07  loss: 2.4340 (2.4823)  acc: 0.0000 (0.0852)  time: 0.0944  data: 0.0001  max mem: 4589
Epoch [01/50]  [  250/11250]  eta: 0:18:21  lr: 1.11e-06  loss: 2.4410 (2.4874)  acc: 0.1250 (0.0842)  time: 0.0945  data: 0.0001  max mem: 4589
Epoch [01/50]  [  300/11250]  eta: 0:18:06  lr: 1.33e-06  loss: 2.4250 (2.4873)  acc: 0.0000 (0.0814)  time: 0.0945  data: 0.0001 

## 9 · Curve di Apprendimento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, len(history["train_loss"]) + 1)
axes[0].plot(ep, history["train_loss"], label="train")
axes[0].plot(ep, history["val_loss"],   label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, history["train_acc"], label="train")
axes[1].plot(ep, history["val_acc"],   label="val")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle(f"EViT {CFG['backbone']} — keep={CFG['base_keep_rate']} fuse={CFG['fuse_token']}")
plt.tight_layout()
plt.savefig(CFG["output_dir"] / "training_curves.png", dpi=150)
plt.show()

## 10 · Valutazione Test Set

In [ ]:
model.load_state_dict(
    torch.load(CFG["output_dir"] / "best_model.pt", map_location=device)
)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Test"):
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(imgs.to(device))
        all_preds.append(logits.float().argmax(1).cpu())
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

f1_macro = float(f1_score(all_labels, all_preds, average="macro", zero_division=0))
acc      = float((all_preds == all_labels).mean())
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## 11 · Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds, normalize="true")
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix \u2014 EViT {CFG['backbone']} ({CFG['dataset_name']})")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(CFG["output_dir"] / "confusion_matrix.png", dpi=150)
wandb.log({"confusion_matrix": wandb.Image(fig)})
plt.show()

## 12 · Token Budget

In [ ]:
num_patches = (CFG["img_size"] // CFG["patch_size"]) ** 2
remaining   = [num_patches]
fuse_extra  = 1 if CFG["fuse_token"] else 0
for kr in CFG["keep_rate"]:
    prev = remaining[-1]
    remaining.append(math.ceil(prev * kr) + fuse_extra if kr < 1.0 else prev)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(len(remaining)), remaining, marker="o", linewidth=2)
ax.axhline(num_patches, linestyle="--", color="gray",
           label=f"Token iniziali ({num_patches})")
for loc in CFG["drop_loc"]:
    ax.axvline(loc + 1, linestyle=":", color="red", alpha=0.5, label=f"Pruning @{loc}")
ax.set_xlabel("Dopo il blocco"); ax.set_ylabel("Token rimanenti")
ax.set_title(f"Token Budget EViT (keep={CFG['base_keep_rate']}, fuse={CFG['fuse_token']})")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CFG["output_dir"] / "token_budget.png", dpi=150)
plt.show()
print(f"Token iniziali : {num_patches}")
print(f"Token finali   : {remaining[-1]}")
print(f"Compressione   : {remaining[-1]/num_patches:.1%}")

## 13 · WandB log finale

In [ ]:
wandb.log({"test/accuracy": acc, "test/f1_macro": f1_macro})
wandb.summary.update({"test/accuracy": acc, "test/f1_macro": f1_macro})
wandb.finish()
print(f"Test accuracy : {acc:.4f}  |  F1 macro : {f1_macro:.4f}")